# `pipeline` -- Win-Probability & Score-Prediction Pipeline

This is the **default, recommended path** of the whole merged project: the pre-match model, dynamic 2nd/1st-innings models, phase-specific evaluation, the score regression zoo, and the locked-2026-holdout external evaluation.

Ported from `project_gagan/notebooks/ipl_pipeline_2008_2026.ipynb` (cells 4, 6, 9, 12, 14, 16, 26, 28, 32, 41, 49-58), rewritten as small, independently-testable functions built on top of the already-tested `data`/`elo`/`features`/`metrics`/`models` notebooks instead of inlining that logic again.

**A bug fix versus the original notebook is baked into this design**, not just patched after the fact: the original notebook's cell 47 re-printed pre-match Cal. LR metrics using a *stale* `p_lr` variable that had been silently reassigned by an intervening cell, producing nonsense numbers (Brier=0.81, AUC=0.03) that flatly contradicted the correct values computed earlier in the same run (see `../DEEP_COMPARISON.md`, "Notable discrepancies", for the full forensic trail). Every function below returns its results in a **dict**, and any caller printing a summary reads directly from that dict -- there is no module-level/notebook-global mutable state for a later cell to accidentally overwrite, so this entire *class* of bug structurally cannot recur here.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import beta as beta_dist
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

from src.data import ABBREV, load_ball_by_ball, build_match_table
from src.elo import compute_elo
from src.features import compute_form_h2h, compute_h2h_beta
from src.metrics import brier_skill_score, ece
from src.models import make_score_zoo

## Module-level constants: feature lists and the global seed

- `PRE_FEAT`: the 5 features used by every pre-match model --   Elo difference, recent-form difference, raw head-to-head rate,   and the two toss-decision flags.
- `DYN2`: the 7 delivery-level features used by the **2nd-innings   (chasing team)** dynamic model -- runs still needed, balls   left, wickets left, current run rate, required run rate, Elo   advantage, and match phase.
- `DYN1`: the 7 delivery-level features used by the **1st-innings   (batting team)** dynamic model -- current score/wickets, balls   left, current run rate, projected final total, Elo advantage,   and phase.
- `SEED = 42`: threaded through every `random_state` in this   module for full reproducibility.

In [2]:
PRE_FEAT = ["elo_diff", "form_diff", "h2h", "toss_bat_first", "toss_field_first"]
DYN2 = ["runs_needed", "balls_remaining", "wkts_remaining", "crr", "rrr", "elo_adv", "phase"]
DYN1 = ["team_runs", "team_wicket", "balls_remaining", "run_rate", "proj_total", "elo_adv", "phase"]
SEED = 42

## `load_and_prepare()` -- Step 0: everything downstream starts here

The single function every entry point (`run_all.py`, `notebooks/run_all.ipynb`) calls first:
1. Load and clean the ball-by-ball data (`load_ball_by_ball`).
2. Aggregate to one row per match (`build_match_table`).
3. Sort chronologically (**required** before Elo/form/H2H, both    of which assume match order).
4. Attach Elo ratings, rolling form, raw head-to-head, and the    Beta-smoothed head-to-head -- all four of `elo`/`features`'s    walk-forward functions in one place.

Returns both the raw delivery-level `df` (needed later for dynamic/delivery-level features) and the match-level `match_df` (needed for the pre-match model) as a pair -- callers use whichever they need.

In [3]:
def load_and_prepare(xlsx_path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load ball-by-ball data, build the match table, and attach Elo/form/H2H."""
    df = load_ball_by_ball(xlsx_path)
    match_df = build_match_table(df)
    match_df = match_df.sort_values("match_id").reset_index(drop=True)
    match_df, _ = compute_elo(match_df)
    match_df["form1"], match_df["form2"], match_df["h2h"], _, _ = compute_form_h2h(match_df)
    match_df["form_diff"] = match_df["form1"] - match_df["form2"]
    match_df["h2h_beta"] = compute_h2h_beta(match_df)
    return df, match_df

## `train_pre_match_internal()` -- the pre-match model, honestly evaluated

**Internal train/test split**: seasons <=2020 for training, >2020 for testing (347 matches). This is a *time-based* split, not a random one -- essential, since a random split would let the model "see the future" via Elo ratings computed using later matches.

Two models are trained side by side:
- **Cal. LR**: calibrated Logistic Regression
- **Cal. GBT**: calibrated Gradient Boosting (imported locally   inside the function, since it's only needed here)

A **climatology baseline** (always predict the training-set base rate) is computed too, giving the Brier Skill Score something meaningful to compare against.

The inner `_row()` closure computes all 5 metrics (Brier, AUC, accuracy, BSS, ECE) for a given probability array -- defined once, applied to both models' predictions, so the metric definitions can't drift between the two.

**Caveat surfaced directly in the returned numbers**: this pre-match model's AUC hovers around 0.45-0.50 -- essentially random. That's not a bug; T20 pre-match outcomes are genuinely close to a coin flip from these features alone (see `../docs/known_limitations.md`).

In [4]:
def train_pre_match_internal(match_df: pd.DataFrame) -> dict:
    """Internal pre-match evaluation: train <=2020, test >2020 (cell 16)."""
    train_m = match_df[match_df["year"] <= 2020]
    test_m = match_df[match_df["year"] > 2020]

    sc = StandardScaler()
    X_tr = sc.fit_transform(train_m[PRE_FEAT])
    X_te = sc.transform(test_m[PRE_FEAT])
    y_tr, y_te = train_m["team1_win"].values, test_m["team1_win"].values

    clim_tr = float(y_tr.mean())
    p_clim = np.full(len(y_te), clim_tr)

    cal_lr = CalibratedClassifierCV(LogisticRegression(C=1.0), method="isotonic", cv=5)
    cal_lr.fit(X_tr, y_tr)

    from sklearn.ensemble import GradientBoostingClassifier
    cal_gbt = CalibratedClassifierCV(GradientBoostingClassifier(random_state=SEED), method="isotonic", cv=5)
    cal_gbt.fit(X_tr, y_tr)

    p_lr = cal_lr.predict_proba(X_te)[:, 1]
    p_gbt = cal_gbt.predict_proba(X_te)[:, 1]

    def _row(p):
        return {
            "brier": brier_score_loss(y_te, p),
            "auc": roc_auc_score(y_te, p),
            "acc": float(((p > 0.5).astype(int) == y_te).mean()),
            "bss": brier_skill_score(y_te, p, p_clim),
            "ece": ece(y_te, p),
        }

    return {
        "climatology": {"brier": brier_score_loss(y_te, p_clim), "auc": 0.5, "acc": float(y_te.mean()), "bss": 0.0, "ece": 0.0},
        "cal_lr": _row(p_lr),
        "cal_gbt": _row(p_gbt),
        "sc": sc, "cal_lr_model": cal_lr, "cal_gbt_model": cal_gbt,
        "p_lr": p_lr, "p_gbt": p_gbt, "y_te": y_te,
    }

## `build_dynamic_2nd()` -- delivery-level features, chasing team

Takes the raw delivery-level `df`, filters to 2nd innings only, and computes every `DYN2` feature directly from gagan's raw columns:
- `balls_remaining` / `runs_needed` / `wkts_remaining`: how far   the chase still has to go
- `crr` / `rrr`: current and required run rates
- `phase`: Powerplay/Middle/Death (same formula used throughout   this project)
- `chasing_wins`: the label -- did the batting (chasing) team   actually win?
- `elo_adv`: joins in the match-level Elo ratings computed by   `load_and_prepare`, expressed as the chasing team's advantage   (`elo2 - elo1`, since team2 is always the team that bowled   first / bats second)

In [5]:
def build_dynamic_2nd(df: pd.DataFrame, match_df: pd.DataFrame) -> pd.DataFrame:
    """Delivery-level 2nd-innings features (cell 26)."""
    df2 = df[df["innings"] == 2].copy()
    df2["balls_remaining"] = (df2["overs"] * 6 - df2["team_balls"]).clip(lower=1)
    df2["runs_needed"] = (df2["runs_target"] - df2["team_runs"]).clip(lower=0)
    df2["wkts_remaining"] = (10 - df2["team_wicket"]).clip(lower=0)
    df2["crr"] = df2["team_runs"] / df2["team_balls"].clip(lower=1) * 6
    df2["rrr"] = df2["runs_needed"] / df2["balls_remaining"] * 6
    df2["phase"] = (df2["over"] > 6).astype(int) + (df2["over"] > 15).astype(int)
    df2["chasing_wins"] = df2["batting_wins"]

    elo_map = match_df.set_index("match_id")[["elo1", "elo2"]]
    df2 = df2.join(elo_map, on="match_id", how="left")
    df2["elo_adv"] = df2["elo2"] - df2["elo1"]
    return df2

## `build_dynamic_1st()` -- delivery-level features, batting-first team

The mirror image of `build_dynamic_2nd`, but there's no chase to track (no target exists yet in the 1st innings), so this function instead computes `proj_total` -- a simple linear projection of the final score based on the current run rate. `defending_wins` is the label: will the team currently batting go on to successfully defend this total? `elo_adv` here is `elo1 - elo2` (the batting-first team's advantage, the mirror of the 2nd-innings version above).

In [6]:
def build_dynamic_1st(df: pd.DataFrame, match_df: pd.DataFrame) -> pd.DataFrame:
    """Delivery-level 1st-innings features (cell 28)."""
    df1 = df[df["innings"] == 1].copy()
    df1["balls_remaining"] = (df1["overs"] * 6 - df1["team_balls"]).clip(lower=1)
    df1["run_rate"] = df1["team_runs"] / df1["team_balls"].clip(lower=1) * 6
    df1["proj_total"] = df1["team_runs"] + df1["run_rate"] * df1["balls_remaining"] / 6
    df1["phase"] = (df1["over"] > 6).astype(int) + (df1["over"] > 15).astype(int)
    df1["defending_wins"] = df1["batting_wins"]

    elo_map = match_df.set_index("match_id")[["elo1", "elo2"]]
    df1 = df1.join(elo_map, on="match_id", how="left")
    df1["elo_adv"] = df1["elo1"] - df1["elo2"]
    return df1

## `train_dynamic_internal()` -- the recommended default win-probability models

**This is the headline result of the whole merged project**: calibrated Logistic Regression trained on `DYN2`/`DYN1` features, evaluated on the same internal 2021-2025 test window as the pre-match model. Verified live: **AUC 0.878** for the 2nd-innings (chasing) model -- dramatically better than the pre-match model's ~0.50, because by the 2nd innings there's an actual target and match state to reason about.

The function trains both models (2nd innings on `DYN2`, 1st innings on `DYN1`) and returns not just their metrics but also the fitted scalers/models/train-test splits themselves -- `phase_specific_eval` below needs `train2`/`test2`/`dsc2` to build its own phase-restricted sub-models.

In [7]:
def train_dynamic_internal(df2: pd.DataFrame, df1: pd.DataFrame) -> dict:
    """Internal dynamic 2nd/1st-innings evaluation (cells 26, 28)."""
    train2, test2 = df2[df2["year"] <= 2020], df2[df2["year"] > 2020]
    dsc2 = StandardScaler()
    dyn2_model = CalibratedClassifierCV(LogisticRegression(C=1.0, max_iter=500), method="isotonic", cv=5)
    dyn2_model.fit(dsc2.fit_transform(train2[DYN2]), train2["chasing_wins"].values)
    p2 = dyn2_model.predict_proba(dsc2.transform(test2[DYN2]))[:, 1]
    y2 = test2["chasing_wins"].values

    train1, test1 = df1[df1["year"] <= 2020], df1[df1["year"] > 2020]
    dsc1 = StandardScaler()
    dyn1_model = CalibratedClassifierCV(LogisticRegression(C=1.0, max_iter=500), method="isotonic", cv=5)
    dyn1_model.fit(dsc1.fit_transform(train1[DYN1]), train1["defending_wins"].values)
    p1 = dyn1_model.predict_proba(dsc1.transform(test1[DYN1]))[:, 1]
    y1 = test1["defending_wins"].values

    return {
        "dyn2": {"brier": brier_score_loss(y2, p2), "auc": roc_auc_score(y2, p2),
                 "acc": float(((p2 > 0.5).astype(int) == y2).mean()), "ece": ece(y2, p2)},
        "dyn1": {"brier": brier_score_loss(y1, p1), "auc": roc_auc_score(y1, p1),
                 "acc": float(((p1 > 0.5).astype(int) == y1).mean())},
        "dsc2": dsc2, "dyn2_model": dyn2_model, "train2": train2, "test2": test2,
        "dsc1": dsc1, "dyn1_model": dyn1_model,
    }

## `phase_specific_eval()` -- a separate model per match phase

**Empirically justified design choice**: rather than one global 2nd-innings model, this trains three *independent* calibrated models -- one each restricted to Powerplay/Middle/Death overs -- and evaluates each on its own phase's held-out data. The result is a striking gradient: Powerplay AUC 0.821, Middle 0.886, **Death 0.953**. The closer the match gets to the end, the more predictable it becomes, which is intuitively exactly right and empirically supports splitting by phase rather than relying on the `phase` feature alone inside one combined model.

Implementation detail: `dsc2` (the 2nd-innings scaler, already fit on the *full* `DYN2` feature set by `train_dynamic_internal`) is reused here rather than re-fit -- the phase-specific sub-models just select a subset of already-scaled columns (`idxs`), since `phase` itself is dropped as a feature (it's now constant within each phase-restricted subset, so it carries no information).

In [8]:
def phase_specific_eval(train2: pd.DataFrame, test2: pd.DataFrame, dsc2: StandardScaler) -> dict:
    """Separate calibrated model per match phase (Powerplay/Middle/Death), cell 32."""
    dyn2_no_phase = [f for f in DYN2 if f != "phase"]
    idxs = [DYN2.index(f) for f in dyn2_no_phase]
    phase_names = ["Powerplay (0-6 ov)", "Middle (6-15 ov)", "Death (15-20 ov)"]
    results = {}
    for pid, pname in enumerate(phase_names):
        tr_p = train2[train2["phase"] == pid]
        te_p = test2[test2["phase"] == pid]
        m_p = CalibratedClassifierCV(LogisticRegression(C=1.0, max_iter=500), method="isotonic", cv=5)
        m_p.fit(dsc2.transform(tr_p[DYN2])[:, idxs], tr_p["chasing_wins"].values)
        pp = m_p.predict_proba(dsc2.transform(te_p[DYN2])[:, idxs])[:, 1]
        yp = te_p["chasing_wins"].values
        results[pname] = {"brier": brier_score_loss(yp, pp), "auc": roc_auc_score(yp, pp), "n": len(yp)}
    return results

## `train_score_zoo_and_save()` -- final-score regression

Trains **three separate model zoos** (each the same 5-regressor family from `make_score_zoo()`, but with a fresh fit): one for pre-match score prediction, one for 1st-innings score at various points in the innings, one for 2nd-innings. All three are saved into a single joblib bundle at `models/ipl_score_pipeline.pkl` (this fixes DEF-C02 from the original project: the canonical, reported score pipeline is this sklearn-based artefact, kept distinct from project_gagan's supplementary Keras MLP notebook, which was **not** ported into this merge since it isn't part of the pipeline's reported headline results).

The private `_fit_inn_zoo` closure is reused for both the 1st- and 2nd-innings cases (only the feature-column list and dataframe differ), avoiding duplicating the same fit/evaluate loop twice.

**Expected result, not a bug**: pre-match score R² comes out negative for every model -- predicting a final score with zero in-game information is worse than just guessing the historical average score. This mirrors project_gagan's own original finding exactly (see `../docs/known_limitations.md`).

In [9]:
def train_score_zoo_and_save(df1: pd.DataFrame, df2: pd.DataFrame, match_df: pd.DataFrame, out_path: str) -> dict:
    """
    Pre-match + 1st/2nd-innings score regressor zoo, saved via joblib
    (fixes DEF-C02: this is the canonical score pipeline artefact, distinct
    from project_gagan's supplementary MLP notebook, which was not ported).
    """
    import joblib

    inn1_final = df1.groupby("match_id")["team_runs"].max().reset_index().rename(columns={"team_runs": "final_score"})
    inn2_final = df2.groupby("match_id")["team_runs"].max().reset_index().rename(columns={"team_runs": "final_score"})

    pm = match_df[["match_id", "year"] + PRE_FEAT + ["score1"]].dropna()
    train_pm, test_pm = pm[pm["year"] <= 2020], pm[pm["year"] > 2020]
    sc_pre = StandardScaler()
    X_tr_pm = sc_pre.fit_transform(train_pm[PRE_FEAT])
    X_te_pm = sc_pre.transform(test_pm[PRE_FEAT])
    pm_zoo = make_score_zoo()
    pm_metrics = {}
    for name, mdl in pm_zoo.items():
        mdl.fit(X_tr_pm, train_pm["score1"].values)
        pred = mdl.predict(X_te_pm)
        pm_metrics[name] = {
            "MAE": float(mean_absolute_error(test_pm["score1"], pred)),
            "R2": float(r2_score(test_pm["score1"], pred)),
        }

    def _fit_inn_zoo(df_inn, feat_cols, final_df):
        merged = df_inn.merge(final_df, on="match_id")
        train_i, test_i = merged[merged["year"] <= 2020], merged[merged["year"] > 2020]
        sc = StandardScaler()
        X_tr = sc.fit_transform(train_i[feat_cols])
        X_te = sc.transform(test_i[feat_cols])
        zoo = make_score_zoo()
        metrics = {}
        for name, mdl in zoo.items():
            mdl.fit(X_tr, train_i["final_score"].values)
            pred = mdl.predict(X_te)
            metrics[name] = {
                "MAE": float(mean_absolute_error(test_i["final_score"], pred)),
                "R2": float(r2_score(test_i["final_score"], pred)),
            }
        return zoo, sc, metrics

    inn1_zoo, sc_inn1, inn1_metrics = _fit_inn_zoo(df1, DYN1, inn1_final)
    inn2_zoo, sc_inn2, inn2_metrics = _fit_inn_zoo(df2, DYN2, inn2_final)

    bundle = {
        "pm_zoo": pm_zoo, "sc_pre": sc_pre,
        "inn1_zoo": inn1_zoo, "sc_inn1": sc_inn1,
        "inn2_zoo": inn2_zoo, "sc_inn2": sc_inn2,
    }
    joblib.dump(bundle, out_path)
    return {"pre_match": pm_metrics, "inn1": inn1_metrics, "inn2": inn2_metrics}

## `retrain_pre_match_full()` -- required before touching the 2026 holdout

**Why this function has to exist separately from `train_pre_match_internal`**: the internal model above is trained on seasons <=2020 only, so it deliberately throws away 2021-2025 signal. Evaluating a locked, never-seen 2026 holdout with that internal-only model would understate real-world performance -- so before touching the 2026 data, the pre-match model must be **retrained on the entire historical dataset (2008-2025)**. This exact ordering mistake was caught and fixed during the initial port: an earlier draft of `run_all.py` accidentally reused the internal-only model here, producing 58.1% instead of the correct, verified 64.9% accuracy on the 2026 holdout.

In [10]:
def retrain_pre_match_full(match_df: pd.DataFrame) -> tuple[CalibratedClassifierCV, StandardScaler]:
    """Retrain the pre-match model on the FULL 2008-2025 dataset (cell 49) --
    required before evaluating on the locked 2026 holdout. Using the
    internal-only (train<=2020) model here would understate real-world
    performance, since it throws away 2021-2025 training signal."""
    sc_pre = StandardScaler()
    pre_model = CalibratedClassifierCV(LogisticRegression(C=1.0), method="isotonic", cv=5)
    pre_model.fit(sc_pre.fit_transform(match_df[PRE_FEAT]), match_df["team1_win"].values)
    return pre_model, sc_pre

## `evaluate_2026_pre_match()` -- the locked, honest external test

This is the single most important integrity check in the whole project: 74 IPL 2026 matches whose results were **never touched during any of the training/tuning above**. It's a genuine out-of-sample test of real-world generalisation.

**Mechanics:**
1. Load the 2026 pre-match CSV, translating short team codes    (`RCB`, `MI`, ...) back to full names via `ABBREV`.
2. Recompute `toss_bat_first`/`toss_field_first` and derive the    actual match winner from `bat_first_won` -- with one hardcoded    correction for `match_id == 12` (a specific known data-quality    fix inherited from project_gagan's original notebook, not    something introduced by this merge).
3. **Walk forward through the 2026 season match by match**,    continuing the exact same Elo/form/H2H bookkeeping that    `load_and_prepare` built up through 2025 -- each 2026 match    is predicted using only information available *before* it    was played, then the actual result updates the running Elo/   form/H2H state for the *next* 2026 match. This is the same    walk-forward principle as `compute_form_h2h`, just extended    one season further using real, not synthetic, future data.
4. Report accuracy with a proper **Clopper-Pearson exact 95%    confidence interval** (via the Beta distribution, not a    normal approximation, which would be inappropriate for a    binomial proportion with only 74 trials), a p-value against    the 50/50 coin-flip null hypothesis, and a genuinely    independent "naive" baseline (always pick whichever side won    more often in the 2026 data).

**Verified result**: 64.9% accuracy (48/74), 95% CI [52.9%, 75.6%], p=0.011 -- nominally better than the naive 63.5% baseline, but see `../docs/known_limitations.md` for why this shouldn't be over-interpreted (the CI is wide, and the underlying pre-match AUC is ~0.51).

In [11]:
def evaluate_2026_pre_match(match_df: pd.DataFrame, pre_model, sc_pre, pm26_path: str) -> dict:
    """Evaluate a pre-match model (already retrained on full 2008-2025 data
    via retrain_pre_match_full) on the locked 2026 holdout (cells 51, 53, 54)."""
    _, elo_2025 = compute_elo(match_df)
    _, _, _, form_hist_full, h2h_hist_full = compute_form_h2h(match_df)

    pm26 = pd.read_csv(pm26_path)
    for col in ["team_bat_first", "team_bowl_first", "toss_winner", "match_winner"]:
        if col in pm26.columns:
            pm26[col] = pm26[col].map(lambda x: ABBREV.get(str(x).strip(), x) if pd.notna(x) else np.nan)
    pm26["toss_bat_first"] = ((pm26["toss_winner"] == pm26["team_bat_first"]) & (pm26["toss_decision"] == "Bat")).fillna(0).astype(int)
    pm26["toss_field_first"] = ((pm26["toss_winner"] == pm26["team_bowl_first"]) & (pm26["toss_decision"] == "Bowl")).fillna(0).astype(int)
    pm26["actual_winner"] = np.where(pm26["bat_first_won"] == 1, pm26["team_bat_first"], pm26["team_bowl_first"])
    pm26.loc[pm26["match_id"] == 12, "actual_winner"] = pm26.loc[pm26["match_id"] == 12, "team_bowl_first"].values[0]

    WINDOW, K = 5, 32
    elo26 = dict(elo_2025)
    form26 = {k: list(v) for k, v in form_hist_full.items()}
    h2h26 = {k: dict(v) for k, v in h2h_hist_full.items()}

    pre_results = []
    for _, r in pm26.sort_values("match_id").iterrows():
        t1, t2, mid = r["team_bat_first"], r["team_bowl_first"], r["match_id"]
        e1, e2 = elo26.get(t1, 1500.0), elo26.get(t2, 1500.0)
        elo_diff = e1 - e2
        h1, h2 = form26.get(t1, []), form26.get(t2, [])
        form_diff = (np.mean(h1[-WINDOW:]) if h1 else 0.5) - (np.mean(h2[-WINDOW:]) if h2 else 0.5)
        key = frozenset([t1, t2])
        if key not in h2h26:
            h2h26[key] = {t1: 0, t2: 0, "n": 0}
        e = h2h26[key]
        h2h_rate = e.get(t1, 0) / e["n"] if e["n"] > 0 else 0.5
        feat = np.array([[elo_diff, form_diff, h2h_rate, r["toss_bat_first"], r["toss_field_first"]]])
        p_bat = pre_model.predict_proba(sc_pre.transform(feat))[0, 1]
        pred = t1 if p_bat > 0.5 else t2
        actual = r["actual_winner"]
        pre_results.append({"match_id": mid, "bat_first": t1, "bowl_first": t2, "pred": pred, "actual": actual, "correct": pred == actual})

        t1_won = int(t1 == actual)
        exp1 = 1 / (1 + 10 ** ((e2 - e1) / 400))
        delta = K * (t1_won - exp1)
        elo26[t1] += delta
        elo26[t2] -= delta
        form26.setdefault(t1, []).append(t1_won)
        form26.setdefault(t2, []).append(1 - t1_won)
        e[t1 if t1_won else t2] += 1
        e["n"] += 1

    pre_df = pd.DataFrame(pre_results)
    n = len(pre_df)
    k = int(pre_df["correct"].sum())
    acc = k / n
    ci_lo = beta_dist.ppf(0.025, k, n - k + 1)
    ci_hi = beta_dist.ppf(0.975, k + 1, n - k)
    z_stat = (acc - 0.5) / (0.5 / n ** 0.5)
    p_val = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    bat_first_win_rate = (pre_df["actual"] == pre_df["bat_first"]).mean()
    naive_acc = max(bat_first_win_rate, 1.0 - bat_first_win_rate)

    return {
        "accuracy": acc, "k": k, "n": n, "ci": (ci_lo, ci_hi), "p_value": p_val,
        "naive_baseline_acc": naive_acc, "pm26": pm26, "elo_2025": elo_2025,
        "pre_df": pre_df,
    }